# 17. 예보 연동 지원 대시보드 (HTML)

16번 결과를 읽어 **인터넷 없이 열리는 HTML 파일 하나**를 만든다. 화면은 두 개다.

| 화면 | 사용자 | 내용 | 공개 |
|---|---|---|---|
| 관광객 화면 | 춘천 방문 예정 관광객 | 예보 강수량 → 혜택 신호등(평상시/1단계 10mm/2단계 30mm) → 혜택(릴레이 할인·취소 고객 쿠폰 등), 제휴처 후보(공공 상가정보 매장 수), 비 오는 날 코스 | 공개 가능 (분석 수치 없음) |
| 담당자 화면 | 춘천시 담당자 | 예보(강수량·최고기온) → 날씨 판정, 취약 격자 지도, 우선 지원 행정동(16 select_priority와 같은 규칙), 07 모델 유형별 예측, 강남 vs 춘천 | **비공개** (대회 데이터로 만든 격자 값 포함) |

**먼저 실행할 것**: 16번 노트북 (`16_vulnerable_grid.csv`, `16_grid_rain_forecast.csv` 필요). 제휴처 후보는 `data/processed/stores_grid.csv`(공공 상가정보)에서 셈

**출력**: `outputs/dashboard/dashboard.html` → 드라이브에서 내려받아 더블클릭하면 브라우저에서 열린다. 외부 사이트에 올리지 않는다.

## 1. 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. 대시보드 HTML 만들기
격자 데이터와 행정동 경계를 JSON으로 묶어 HTML 안에 넣는다. 지도는 외부 라이브러리 없이 브라우저 기본 기능(canvas)으로 그린다.

In [ ]:
# ============================================================
# 17-2. 대시보드 HTML 만들기 (관광객 화면 + 담당자 화면)
#   입력: data/processed/stores_grid.csv              (제휴처 후보 매장 수, 공공 상가정보)
#         outputs/tables/16_vulnerable_grid.csv     (격자 등급·06 기반 점수·행정동)
#         outputs/tables/16_grid_rain_forecast.csv  (07 모델 격자 예측 10/30/45mm)
#         data/external/adm/adm_gangnam_chuncheon_2025.geojson (행정동 경계)
#   출력: outputs/dashboard/dashboard.html  (파일 하나, 인터넷 없이 열림)
#   ※ 담당자 화면에는 대회 데이터로 만든 격자 값이 들어가므로 외부에 공개하지 않는다.
# ============================================================
import json
from pathlib import Path
import numpy as np
import pandas as pd
from pyproj import Transformer

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
TAB = BASE / "outputs" / "tables"
GEO = BASE / "data" / "external" / "adm" / "adm_gangnam_chuncheon_2025.geojson"
OUT = BASE / "outputs" / "dashboard"; OUT.mkdir(parents=True, exist_ok=True)

# ---- 1. 격자 데이터 ----
v = pd.read_csv(TAB / "16_vulnerable_grid.csv")
f = pd.read_csv(TAB / "16_grid_rain_forecast.csv")
for d in (v, f):
    d["kx"], d["ky"] = d["X_COORD"].round(2), d["Y_COORD"].round(2)
g = v.merge(f[["region", "kx", "ky", "pred_10mm", "pred_30mm", "pred_45mm"]], on=["region", "kx", "ky"], how="left")
print("격자 수:", g.groupby("region").size().to_dict(), "/ 07 예측 연결 안 된 칸:", int(g["pred_45mm"].isna().sum()))
for c in ["pred_10mm", "pred_30mm", "pred_45mm"]:
    g[c] = g[c].fillna(0)

TYPES = sorted(g["type"].dropna().unique())
cells = {}
for reg, d in g.groupby("region"):
    cells[reg] = {
        "x": d["X_COORD"].round(1).tolist(), "y": d["Y_COORD"].round(1).tolist(),
        "dong": d["dong"].fillna("").tolist(), "type": [TYPES.index(t) for t in d["type"]],
        "n": d["n_stores"].fillna(0).astype(int).tolist(),
        "golf": d["has_golf"].fillna(0).astype(int).tolist(), "lego": d["has_lego"].fillna(0).astype(int).tolist(),
        "heat": d["heat_score"].round(5).tolist(), "rain": d["rain_score"].round(5).tolist(),
        "dbl": d["double_vuln"].astype(bool).astype(int).tolist(),
        "p10": d["pred_10mm"].round(2).tolist(), "p30": d["pred_30mm"].round(2).tolist(), "p45": d["pred_45mm"].round(2).tolist(),
    }

# ---- 2. 행정동 경계 (EPSG:5179로 변환) ----
tf = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)
gj = json.load(open(GEO, encoding="utf-8"))
bounds = {"강남": [], "춘천": []}
for ft in gj["features"]:
    nm = ft["properties"]["adm_nm"]
    reg = "강남" if "강남구" in nm else ("춘천" if "춘천시" in nm else None)
    if reg is None:
        continue
    geom = ft["geometry"]
    for poly in (geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]):
        r = np.array(poly[0]); x, y = tf.transform(r[:, 0], r[:, 1])
        bounds[reg].append({"name": nm.split()[-1], "pts": np.c_[np.round(x), np.round(y)].astype(int).ravel().tolist()})

# ---- 2-1. 제휴처 후보 (관광객 화면용: 공공 상가정보 매장 수만, 분석 수치 없음) ----
st = pd.read_csv(BASE / "data" / "processed" / "stores_grid.csv", low_memory=False)
st = st[st["region"] == "춘천"].copy()
st["dong"] = st["행정동명"].fillna("")
# 수작업으로 추가한 골프장·레고랜드는 행정동명이 비어 있어 격자의 행정동으로 채움
gd = v[v["region"] == "춘천"].set_index(["kx", "ky"])["dong"]
key = list(zip(pd.to_numeric(st["grid_x"], errors="coerce").round(2), pd.to_numeric(st["grid_y"], errors="coerce").round(2)))
fill = pd.Series([gd.get(k, "") for k in key], index=st.index).fillna("")
st.loc[st["dong"] == "", "dong"] = fill[st["dong"] == ""]
print("행정동 빈 매장:", int((st["dong"] == "").sum()))
CATS = [("숙박·레저", ["숙박", "야외레저"], "무료 일정 변경(1단계), 사전 예약분 취소 수수료 면제(2단계)", 1),
        ("카페·제과", ["카페·제과"], "비 오는 날 릴레이 할인, 취소 고객 쿠폰 사용처", 1),
        ("음식점", ["음식점"], "비 오는 날 릴레이 할인, 취소 고객 쿠폰 사용처", 1),
        ("시장·실내", ["식료품소매", "실내여가"], "비 오는 날 릴레이 할인, 비 오는 날 코스", 1)]
partners = []
for cat, grps, benefit, stage in CATS:
    d = st[st["ind_grp"].isin(grps) & (st["dong"] != "")]
    allcnt = d.groupby("dong").size().sort_values(ascending=False)
    cnt = allcnt.head(8)
    if cat == "숙박·레저":   # 골프장·레고랜드가 있는 동은 순위 밖이어도 표시
        extra = d[d["출처"].astype(str).str.startswith("수동추가")]["dong"].unique()
        cnt = allcnt[allcnt.index.isin(set(cnt.index) | set(extra))]
    areas = []
    for dong, n in cnt.items():
        note = ""
        if cat == "숙박·레저":
            k = d[d["dong"] == dong]["출처"].astype(str)
            golf = int(k.str.contains("골프장").sum()); lego = int(k.str.contains("관광시설").sum())
            note = " · ".join(x for x in [f"골프장 {golf}곳" if golf else "", "레고랜드" if lego else ""] if x)
        areas.append({"dong": dong, "n": int(n), "note": note})
    partners.append({"cat": cat, "total": int(len(d)), "benefit": benefit, "stage": stage, "areas": areas})
print("제휴처 후보:", {p["cat"]: p["total"] for p in partners})

DATA = {"types": TYPES, "cells": cells, "bounds": bounds, "partners": partners}

# ---- 3. HTML 템플릿에 데이터 넣기 ----
TEMPLATE = r"""<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>같은 날씨, 다른 피해 · 예보 연동 지원 대시보드</title>
<style>
  :root{--ink:#1d2430;--sub:#5b6472;--line:#d9dee5;--bg:#f5f7fa;--card:#fff;--navy:#1f3864;
        --r1:#c6dbef;--r2:#6baed6;--r3:#08306b;--h1:#fdd0a2;--h2:#f16913;--h3:#7f2704;--gray:#cfd4db;}
  *{box-sizing:border-box}
  body{margin:0;font-family:"Malgun Gothic","맑은 고딕","Apple SD Gothic Neo","Noto Sans KR",sans-serif;color:var(--ink);background:var(--bg)}
  header{background:var(--navy);color:#fff;padding:14px 24px;display:flex;align-items:center;gap:24px;flex-wrap:wrap}
  header h1{font-size:19px;margin:0}
  header .sub{font-size:12px;opacity:.8}
  .tabs{margin-left:auto;display:flex;gap:6px}
  .tabs button{border:1px solid rgba(255,255,255,.5);background:transparent;color:#fff;padding:8px 16px;border-radius:6px;cursor:pointer;font-size:14px}
  .tabs button.on{background:#fff;color:var(--navy);font-weight:bold}
  main{max-width:1280px;margin:0 auto;padding:18px}
  .card{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:16px;margin-bottom:14px}
  .card h2{font-size:16px;margin:0 0 10px}
  .muted{color:var(--sub);font-size:12px}
  .row{display:flex;gap:14px;flex-wrap:wrap}
  .col{flex:1;min-width:280px}
  label{font-size:13px;color:var(--sub)}
  input[type=range]{width:100%}
  .val{font-size:22px;font-weight:bold}
  .seg button{border:1px solid var(--line);background:#fff;padding:6px 14px;cursor:pointer;font-size:13px}
  .seg button:first-child{border-radius:6px 0 0 6px}.seg button:last-child{border-radius:0 6px 6px 0}
  .seg button.on{background:var(--navy);color:#fff;border-color:var(--navy)}
  .banner{border-radius:8px;padding:10px 14px;font-weight:bold;margin-bottom:10px}
  .b-none{background:#eef1f5;color:var(--sub)} .b-rain{background:#deebf7;color:#08306b} .b-heat{background:#fee6ce;color:#7f2704} .b-ok{background:#e5f4e4;color:#1e6b2f}
  .kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:10px}
  .kpi{border:1px solid var(--line);border-radius:8px;padding:10px}
  .kpi .n{font-size:24px;font-weight:bold}
  .kpi .t{font-size:12px;color:var(--sub)}
  #map{width:100%;height:600px;display:block;border:1px solid var(--line);border-radius:8px;background:#f2f4f7}
  .legend{display:flex;gap:14px;flex-wrap:wrap;font-size:12px;margin-top:8px;align-items:center}
  .sw{display:inline-block;width:14px;height:14px;border-radius:2px;vertical-align:middle;margin-right:4px;border:1px solid rgba(0,0,0,.1)}
  table{border-collapse:collapse;width:100%;font-size:13px}
  th,td{border-bottom:1px solid var(--line);padding:6px 8px;text-align:left}
  th{background:#eef2f7;font-weight:bold}
  td.num{text-align:right}
  .grayv{color:#9aa1ab}
  #tip{position:fixed;pointer-events:none;background:rgba(29,36,48,.92);color:#fff;font-size:12px;padding:6px 8px;border-radius:6px;display:none;z-index:10;line-height:1.5}
  /* tourist */
  .lights{display:flex;gap:18px;justify-content:center;margin:12px 0}
  .light{text-align:center;font-size:13px;color:var(--sub)}
  .light .dot{width:64px;height:64px;border-radius:50%;margin:0 auto 6px;background:#e3e6ea;border:3px solid #d0d4da}
  .light.on .dot{border-color:var(--ink)}
  .light.g.on .dot{background:#59b36b}.light.y.on .dot{background:#f2c037}.light.r.on .dot{background:#e0533d}
  .light.on{color:var(--ink);font-weight:bold}
  .benefits{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:10px}
  .bn{border:1px solid var(--line);border-radius:8px;padding:12px}
  .bn.off{opacity:.35}
  .bn b{display:block;margin-bottom:4px}
  .tag{display:inline-block;font-size:11px;padding:2px 6px;border-radius:4px;background:#eef2f7;color:var(--sub);margin-left:6px}
  .hide{display:none}
  .cmp td:nth-child(2),.cmp td:nth-child(3){text-align:center}
</style>
</head>
<body>
<header>
  <div><h1>같은 날씨, 다른 피해 · 예보 연동 지원 대시보드</h1>
  <div class="sub">2026 빅콘테스트 · 프로토타입 (2025년 7~9월 신한카드·SK 유동인구 분석 결과 기반)</div></div>
  <div class="tabs"><button id="tabT" class="on">관광객 화면</button><button id="tabC">담당자 화면</button></div>
</header>

<main>
<!-- ================= 관광객 화면 ================= -->
<section id="viewT">
  <div class="card" style="background:#eaf3ea;border-color:#cfe5cf"><b>☔ 비 와도 괜찮아요, 춘천.</b> 비 예보가 있는 날에는 일정 변경과 실내 혜택을 드립니다.</div>
  <div class="row">
    <div class="col card">
      <h2>1. 여행 날짜와 그날 춘천 예보</h2>
      <label>여행 날짜</label><br><input type="date" id="tDate" style="padding:6px;font-size:14px"><br><br>
      <label>그날 춘천 예보 강수량 (mm)</label>
      <div class="val"><span id="tRainV">0</span> mm</div>
      <input type="range" id="tRain" min="0" max="80" step="1" value="0">
      <div class="seg" style="margin-top:6px"><button data-v="0">맑음 0mm</button><button data-v="15">비 15mm</button><button data-v="45">큰비 45mm</button></div>
      <p class="muted">실제 운영에서는 기상청 예보(여행 전날 18시 발표)가 자동으로 들어갑니다. 지금은 시연용으로 직접 입력합니다.</p>
    </div>
    <div class="col card">
      <h2>2. 혜택 신호등</h2>
      <div class="lights">
        <div class="light g" id="lg"><div class="dot"></div>평상시<br><span class="muted">비 예보 없음</span></div>
        <div class="light y" id="ly"><div class="dot"></div>1단계 혜택<br><span class="muted">10mm 이상</span></div>
        <div class="light r" id="lr"><div class="dot"></div>2단계 혜택<br><span class="muted">30mm 이상</span></div>
      </div>
      <div id="tMsg" class="banner b-none"></div>
    </div>
  </div>
  <div class="card">
    <h2>3. 지금 받을 수 있는 혜택</h2>
    <div class="benefits">
      <div class="bn" id="bn1"><b>무료 일정 변경</b>숙박·레저 예약 날짜를 수수료 없이 바꿀 수 있어요.<span class="tag">1단계부터</span></div>
      <div class="bn" id="bn2"><b>비 오는 날 릴레이 할인</b>같은 날 시내 제휴 매장 영수증을 보여주면 다음 제휴 매장에서 할인돼요.<span class="tag">1단계부터</span></div>
      <div class="bn" id="bn3"><b>취소·변경 고객 시내 쿠폰</b>비 때문에 레저 예약을 취소·변경하면 시내 제휴 매장 쿠폰을 드려요.<span class="tag">1단계부터</span></div>
      <div class="bn" id="bn4"><b>비 오는 날 코스 안내</b>중앙시장(낭만시장) 등 실내 위주 코스를 추천해요.<span class="tag">1단계부터</span></div>
      <div class="bn" id="bn5"><b>예약 취소 수수료 면제</b>출발 기준일 전에 미리 예약한 숙박·레저만 수수료 없이 취소할 수 있어요.<span class="tag">2단계만</span></div>
    </div>
    <p class="muted">할인율과 사전 예약 기준일은 시범 운영에서 정합니다. 사전 예약 조건은 비 예보를 보고 예약했다 취소하는 남용을 막기 위한 장치예요 (참고: 미국 Priceline 2008, 출발 12일 전 예약분에만 우천 환급 적용).</p>
  </div>
  <div class="card">
    <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px">
      <h2 style="margin:0">4. 제휴처 <span class="tag">후보 · 시범 운영 전</span></h2>
      <div class="seg" id="pSeg"></div>
    </div>
    <p class="muted" id="pDesc" style="margin:8px 0"></p>
    <div class="benefits" id="pCards"></div>
    <p class="muted">매장 수는 소상공인시장진흥공단 상가(상권)정보(2025-06 기준, 공공데이터) 기준이며, 실제 제휴 매장이 아니라 모집 대상 후보입니다. 제휴가 정해지면 매장 이름과 예약 링크로 바뀝니다.</p>
  </div>
  <div class="row">
    <div class="col card"><h2>비 오는 날 추천 코스 <span class="tag">예시</span></h2>
      <table><tr><th>시간</th><th>코스</th><th>받을 수 있는 혜택</th></tr>
      <tr><td>오전</td><td>시내 실내 체험·전시 시설</td><td>취소 고객 쿠폰</td></tr>
      <tr><td>점심</td><td>중앙시장(낭만시장) 먹거리</td><td>릴레이 할인 시작</td></tr>
      <tr><td>오후</td><td>시내 카페·제과점</td><td>릴레이 할인</td></tr>
      <tr><td>저녁</td><td>시내 음식점</td><td>릴레이 할인 + 지역화폐 적립</td></tr></table></div>
    <div class="col card"><h2>안내</h2>
      <p class="muted">발동 기준: 기상청 예보, 여행 전날 18시 발표 춘천 강수량. 10mm 이상 1단계, 30mm 이상 2단계.</p>
      <p class="muted">이 화면에는 상권별 분석 수치가 들어가지 않습니다. 혜택이 방문을 늘리는지는 시범 운영으로 검증할 가설입니다.</p></div>
  </div>
</section>

<!-- ================= 담당자 화면 ================= -->
<section id="viewC" class="hide">
  <div class="card">
    <div class="row" style="align-items:flex-end">
      <div class="col"><label>지역</label><br><div class="seg" id="regSeg"><button data-v="춘천" class="on">춘천</button><button data-v="강남">강남</button></div></div>
      <div class="col"><label>내일 예보 강수량</label><div class="val"><span id="cRainV">45</span> mm</div><input type="range" id="cRain" min="0" max="80" value="45"></div>
      <div class="col"><label>내일 예보 최고기온</label><div class="val"><span id="cTmaxV">34</span> ℃</div><input type="range" id="cTmax" min="20" max="40" value="34"></div>
    </div>
  </div>
  <div id="cBanner" class="banner b-none"></div>
  <div class="card"><div class="kpis">
    <div class="kpi"><div class="n" id="k1">-</div><div class="t">지원 대상 격자 (예상 감소 5% 이상)</div></div>
    <div class="kpi"><div class="n" id="k2">-</div><div class="t">고취약 격자 (10% 이상)</div></div>
    <div class="kpi"><div class="n" id="k3">-</div><div class="t">이중 취약 격자 (폭염·호우 모두 5% 이상)</div></div>
    <div class="kpi"><div class="n" id="k4">-</div><div class="t">1순위 행정동</div></div>
  </div></div>
  <div class="row">
    <div class="card" style="flex:1.4;min-width:340px">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px">
        <h2 id="mapTitle" style="margin:0">취약 상권 지도</h2>
        <div style="display:flex;gap:8px;flex-wrap:wrap">
          <div class="seg" id="laySeg"><button data-v="rain">호우</button><button data-v="heat">폭염</button></div>
          <div class="seg" id="zoomSeg"><button data-v="all" class="on">전체</button><button data-v="city">시내 확대</button></div>
        </div>
      </div>
      <canvas id="map"></canvas>
      <div class="legend" id="legend"></div>
      <p class="muted" id="mapNote"></p>
    </div>
    <div class="card" style="flex:1;min-width:320px">
      <h2>우선 지원 행정동 (상위 10)</h2>
      <div id="prio"></div>
      <p class="muted">정렬: 고취약 칸 수 → 이중 취약 칸 수 → 평균 예상 감소율 (16번 노트북 select_priority와 같은 규칙). 예상 감소율은 06 회귀에서 FDR 보정 후 유의한 효과만 반영.</p>
    </div>
  </div>
  <div class="row">
    <div class="col card"><h2>07 예측 모델: 예보 강수량별 상권 유형 평균 예상 매출 변화</h2><div id="fc"></div>
      <p class="muted">07b 시나리오(10·30·45mm)를 격자 점포 구성에 적용한 값. 사이 값은 직선으로 이어 읽고 45mm 초과는 45mm 값. 회색 = ±5% 안팎, 모델 오차 범위라 해석하지 않음.</p></div>
    <div class="col card"><h2>강남 vs 춘천 (2025년 7~9월)</h2>
      <table class="cmp"><tr><th>구분</th><th>강남</th><th>춘천</th></tr>
      <tr><td>호우일 야외레저 매출</td><td>유의한 변화 없음</td><td><b>−53.4%</b></td></tr>
      <tr><td>폭염일 60대 이상 여가·외출</td><td>유의하지 않음</td><td><b>−18.9%</b></td></tr>
      <tr><td>추정 피해</td><td>0원</td><td>호우 50.0억 + 폭염 4.8억</td></tr>
      <tr><td>기후 취약 격자</td><td>0칸</td><td>380칸</td></tr>
      <tr><td>해석</td><td>실내 소비로 흡수 (가능성)</td><td>옮겨갈 곳이 적음</td></tr></table>
      <p class="muted">모두 FDR 보정 후 기준. 강남은 배달·온라인 이동이 관측되지 않아 모니터링 대상으로 둠.</p></div>
  </div>
</section>
</main>
<div id="tip"></div>

<script>
const D = __DATA__;
const $ = (id) => document.getElementById(id);
const ACTION = {"호우 경보":"우천 안심 혜택 2단계 (취소 수수료 면제 추가)","호우 주의":"우천 안심 혜택 1단계 + 상인 사전 알림","폭염":"냉방·그늘막·쉼터, 어르신 외출 시간대 할인"};
const SHORT = {"호우 경보":"2단계 혜택","호우 주의":"1단계 혜택 + 상인 알림","폭염":"냉방·쉼터·외출 할인"};
const fmt = (v, d=1) => (v>0?"+":"") + v.toFixed(d);

// ---------- tabs ----------
$("tabT").onclick = () => { $("tabT").classList.add("on"); $("tabC").classList.remove("on"); $("viewT").classList.remove("hide"); $("viewC").classList.add("hide"); };
$("tabC").onclick = () => { $("tabC").classList.add("on"); $("tabT").classList.remove("on"); $("viewC").classList.remove("hide"); $("viewT").classList.add("hide"); drawAll(); };

// ---------- tourist ----------
function tUpdate(){
  const mm = +$("tRain").value; $("tRainV").textContent = mm;
  const st = mm >= 30 ? 2 : mm >= 10 ? 1 : 0;
  ["lg","ly","lr"].forEach((id,i) => $(id).classList.toggle("on", i===st));
  ["bn1","bn2","bn3","bn4"].forEach(id => $(id).classList.toggle("off", st<1)); $("bn5").classList.toggle("off", st<2); TST = st; drawPartners();
  const m = $("tMsg");
  if(st===0){ m.className="banner b-ok"; m.textContent="비 예보가 없어요. 예약해 두시면, 비 예보가 나올 때 자동으로 혜택이 열립니다."; }
  else if(st===1){ m.className="banner b-rain"; m.textContent="1단계 혜택 발동: 무료 일정 변경 + 비 오는 날 릴레이 할인 + 취소 고객 시내 쿠폰 + 비 오는 날 코스"; }
  else { m.className="banner b-heat"; m.textContent="2단계 혜택 발동: 1단계 혜택 + 숙박·레저 예약 취소 수수료 면제 (사전 예약분)"; }
}

// ---------- partners (tourist) ----------
let TST = 0, PCAT = 0;
function drawPartners(){
  const P = D.partners; if (!P) return;
  $("pSeg").innerHTML = P.map((p,i)=>`<button data-i="${i}" class="${i===PCAT?'on':''}">${p.cat} ${p.total.toLocaleString()}곳</button>`).join("");
  $("pSeg").querySelectorAll("button").forEach(b => b.onclick = () => { PCAT = +b.dataset.i; drawPartners(); });
  const p = P[PCAT];
  const on = p.stage <= TST;
  $("pDesc").innerHTML = `<b>${p.cat}</b> · 적용 혜택: ${p.benefit} ` + (TST===0 ? "<span class='tag'>비 예보가 나오면 열려요</span>" : (on ? "<span class='tag' style='background:#e5f4e4;color:#1e6b2f'>지금 적용</span>" : "<span class='tag'>2단계에서 적용</span>"));
  $("pCards").innerHTML = p.areas.map(a => `<div class="bn ${on?'':'off'}"><b>${a.dong}</b>${p.cat} 후보 ${a.n.toLocaleString()}곳${a.note?`<br><span class="muted">${a.note}</span>`:""}</div>`).join("");
}
$("tRain").oninput = tUpdate;
document.querySelectorAll("#viewT .seg button").forEach(b => b.onclick = () => { $("tRain").value = b.dataset.v; tUpdate(); });
$("tDate").valueAsDate = new Date();
tUpdate();

// ---------- city ----------
let REG = "춘천", LAYER = "rain", ZOOM = "all", layerTouched = false;
function judge(mm, t){ const o=[]; if(mm>=30) o.push("호우 경보"); else if(mm>=10) o.push("호우 주의"); if(t>=33) o.push("폭염"); return o; }
function grade(v){ return v>=0.10?3 : v>=0.05?2 : v>=0.02?1 : 0; }

function selectPriority(mm, t){
  const res = [];
  for (const reg of ["춘천","강남"]) {
    const c = D.cells[reg];
    for (const lv of judge(mm, t)) {
      const col = lv==="폭염" ? c.heat : c.rain;
      const by = {};
      for (let i=0;i<col.length;i++){
        if (col[i] < 0.05 || !c.dong[i]) continue;
        const k = c.dong[i];
        const o = by[k] || (by[k] = {reg, dong:k, lv, hi:0, dbl:0, n:0, sum:0, gl:0, types:{}});
        o.hi += col[i]>=0.10 ? 1:0; o.dbl += c.dbl[i]; o.n += 1; o.sum += Math.round(col[i]*1000)/10; o.gl += c.golf[i] + c.lego[i];
        o.types[c.type[i]] = (o.types[c.type[i]]||0) + 1;
      }
      Object.values(by).forEach(o => res.push(o));
    }
  }
  res.forEach(o => { o.avg = o.sum/o.n; o.top = D.types[+Object.entries(o.types).sort((a,b)=>b[1]-a[1] || a[0]-b[0])[0][0]]; });
  res.sort((a,b) => b.hi-a.hi || b.dbl-a.dbl || b.avg-a.avg || (a.reg<b.reg?-1:a.reg>b.reg?1:0) || (a.dong<b.dong?-1:a.dong>b.dong?1:0) || (a.lv<b.lv?-1:a.lv>b.lv?1:0));
  return res.slice(0,10);
}

function interp(i, c, mm){
  const xs=[0,10,30,45], ys=[0, c.p10[i], c.p30[i], c.p45[i]], x=Math.min(Math.max(mm,0),45);
  for (let k=1;k<4;k++) if (x<=xs[k]) return ys[k-1] + (ys[k]-ys[k-1])*(x-xs[k-1])/(xs[k]-xs[k-1]);
  return ys[3];
}

// ---------- map ----------
const cv = $("map"), ctx = cv.getContext("2d");
let view = null, grid = null;
function extent(){
  const c = D.cells[REG];
  let xs = c.x.slice(), ys = c.y.slice();
  D.bounds[REG].forEach(b => { for (let i=0;i<b.pts.length;i+=2){ xs.push(b.pts[i]); ys.push(b.pts[i+1]); } });
  if (ZOOM==="city" && REG==="춘천"){
    const q = (a,p) => { const s=a.slice().sort((u,v)=>u-v); return s[Math.floor(p*(s.length-1))]; };
    const low = D.types.indexOf("낮·고령형"), ux=[], uy=[];
    for (let i=0;i<c.x.length;i++) if (c.type[i]!==low){ ux.push(c.x[i]); uy.push(c.y[i]); }
    let bx0=q(ux,.02), bx1=q(ux,.98), by0=q(uy,.02), by1=q(uy,.98);
    for (let i=0;i<c.x.length;i++) if (c.lego[i]){ bx0=Math.min(bx0,c.x[i]); bx1=Math.max(bx1,c.x[i]); by0=Math.min(by0,c.y[i]); by1=Math.max(by1,c.y[i]); }
    return [bx0-400, bx1+400, by0-400, by1+400];
  }
  return [Math.min(...xs), Math.max(...xs), Math.min(...ys), Math.max(...ys)];
}
function colorFor(v, layer){
  const g = grade(v);
  if (g===0) return "#d5d9df";
  return layer==="heat" ? ["","#fdd0a2","#f16913","#7f2704"][g] : ["","#c6dbef","#6baed6","#08306b"][g];
}
function drawMap(){
  const W = cv.clientWidth, H = cv.clientHeight, dpr = window.devicePixelRatio||1;
  cv.width = W*dpr; cv.height = H*dpr; ctx.setTransform(dpr,0,0,dpr,0,0); ctx.clearRect(0,0,W,H);
  const [x0,x1,y0,y1] = extent(), pad=14;
  const s = Math.min((W-2*pad)/(x1-x0), (H-2*pad)/(y1-y0));
  const ox = pad + ((W-2*pad) - (x1-x0)*s)/2, oy = pad + ((H-2*pad) - (y1-y0)*s)/2;
  const X = x => ox + (x-x0)*s, Y = y => H - (oy + (y-y0)*s);
  view = {X, Y, s};
  ctx.save(); ctx.beginPath(); ctx.rect(0,0,W,H); ctx.clip();
  // boundaries
  ctx.fillStyle = "#ffffff"; ctx.strokeStyle = "#333"; ctx.lineWidth = 0.7;
  D.bounds[REG].forEach(b => { ctx.beginPath(); for (let i=0;i<b.pts.length;i+=2){ const px=X(b.pts[i]), py=Y(b.pts[i+1]); i? ctx.lineTo(px,py):ctx.moveTo(px,py);} ctx.closePath(); ctx.fill(); ctx.stroke(); });
  // cells
  const c = D.cells[REG], col = LAYER==="heat" ? c.heat : c.rain;
  const active = layerActive();
  const sz = Math.max(3, 50*s);
  const order = [...col.keys()].sort((a,b)=>col[a]-col[b]);
  grid = [];
  for (const i of order){
    const v = active ? col[i] : 0;
    ctx.fillStyle = colorFor(v, LAYER);
    const px = X(c.x[i]), py = Y(c.y[i]);
    ctx.fillRect(px-sz/2, py-sz/2, sz, sz);
    grid.push([px,py,i]);
  }
  // golf / lego markers
  for (let i=0;i<c.x.length;i++){
    if (!c.golf[i] && !c.lego[i]) continue;
    const px = X(c.x[i]), py = Y(c.y[i]);
    ctx.fillStyle = "#e0533d"; ctx.strokeStyle="#fff"; ctx.lineWidth=1;
    ctx.beginPath();
    if (c.lego[i]) { for(let k=0;k<10;k++){ const r=k%2?3.5:8, a=-Math.PI/2+k*Math.PI/5; ctx.lineTo(px+r*Math.cos(a), py+r*Math.sin(a)); } }
    else { ctx.moveTo(px,py-6); ctx.lineTo(px+5.5,py+4); ctx.lineTo(px-5.5,py+4); }
    ctx.closePath(); ctx.fill(); ctx.stroke();
  }
  ctx.restore();
}
function layerActive(){
  const lv = judge(+$("cRain").value, +$("cTmax").value);
  return LAYER==="heat" ? lv.includes("폭염") : lv.some(l=>l.startsWith("호우"));
}
function legend(){
  const pal = LAYER==="heat" ? ["#fdd0a2","#f16913","#7f2704"] : ["#c6dbef","#6baed6","#08306b"];
  const w = LAYER==="heat" ? "폭염" : "호우";
  $("legend").innerHTML =
    `<b>격자 색 = ${w}일 예상 매출 감소율 (06 기준 등급)</b>`+
    `<span><span class="sw" style="background:#d5d9df"></span>일반 2% 미만 / 해당 날씨 예보 없음</span>`+
    `<span><span class="sw" style="background:${pal[0]}"></span>관심 2~5%</span>`+
    `<span><span class="sw" style="background:${pal[1]}"></span>취약 5~10%</span>`+
    `<span><span class="sw" style="background:${pal[2]}"></span>고취약 10% 이상</span>`+
    `<span><span style="color:#e0533d">▲</span> 골프장 <span style="color:#e0533d">★</span> 레고랜드</span>`+
    `<span>검은 선 = 행정동 경계</span>`;
}

// tooltip
cv.addEventListener("mousemove", e => {
  if (!grid) return;
  const r = cv.getBoundingClientRect(), mx = e.clientX-r.left, my = e.clientY-r.top;
  let best=-1, bd=64;
  for (const [px,py,i] of grid){ const d=(px-mx)**2+(py-my)**2; if (d<bd){bd=d;best=i;} }
  const tip = $("tip");
  if (best<0){ tip.style.display="none"; return; }
  const c = D.cells[REG], mm = +$("cRain").value;
  const p = interp(best, c, mm);
  tip.innerHTML = `<b>${c.dong[best]||"-"}</b> · ${D.types[c.type[best]]}<br>가게 ${c.n[best]}곳${c.golf[best]?" · 골프장":""}${c.lego[best]?" · 레고랜드":""}<br>`+
    `호우 예상 감소 (06): ${(c.rain[best]*100).toFixed(1)}%<br>폭염 예상 감소 (06): ${(c.heat[best]*100).toFixed(1)}%<br>`+
    `07 모델 ${mm}mm 예측: <span class="${Math.abs(p)<5?'grayv':''}">${fmt(p)}%</span>${Math.abs(p)<5?" (오차 범위 내)":""}`;
  tip.style.left = (e.clientX+14)+"px"; tip.style.top = (e.clientY+10)+"px"; tip.style.display="block";
});
cv.addEventListener("mouseleave", () => $("tip").style.display="none");

// ---------- panels ----------
function drawAll(){
  const mm = +$("cRain").value, t = +$("cTmax").value;
  $("cRainV").textContent = mm; $("cTmaxV").textContent = t;
  const lv = judge(mm, t);
  if (!layerTouched) LAYER = lv.some(l=>l.startsWith("호우")) || !lv.includes("폭염") ? "rain" : "heat";
  document.querySelectorAll("#laySeg button").forEach(b => b.classList.toggle("on", b.dataset.v===LAYER));
  document.querySelectorAll("#regSeg button").forEach(b => b.classList.toggle("on", b.dataset.v===REG));
  document.querySelectorAll("#zoomSeg button").forEach(b => { b.classList.toggle("on", b.dataset.v===ZOOM); b.disabled = REG==="강남" && b.dataset.v==="city"; });
  // banner
  const bn = $("cBanner");
  if (REG==="강남"){ bn.className="banner b-ok"; bn.textContent = "강남: 모니터링 중 · 현재 통계적으로 유의한 날씨 피해 없음 (FDR 보정 기준)" + (lv.length? ` — 예보 판정: ${lv.join(", ")}`:""); }
  else if (!lv.length){ bn.className="banner b-none"; bn.textContent="예보 판정: 해당 없음 (강수 10mm 미만, 최고기온 33℃ 미만) → 지원 발동 없음"; }
  else { bn.className = "banner " + (lv.some(l=>l.startsWith("호우"))?"b-rain":"b-heat"); bn.textContent = "예보 판정: " + lv.join(" + ") + " → " + lv.map(l=>ACTION[l]).join(" / "); }
  // KPIs (current region & layer)
  const c = D.cells[REG], col = LAYER==="heat" ? c.heat : c.rain, act = layerActive();
  let k1=0,k2=0,k3=0; for (let i=0;i<col.length;i++){ if(act && col[i]>=0.05) k1++; if(act && col[i]>=0.10) k2++; if(c.dbl[i]) k3++; }
  $("k1").textContent = k1.toLocaleString()+"칸"; $("k2").textContent = k2+"칸"; $("k3").textContent = k3+"칸";
  const pr = selectPriority(mm, t).filter(o=>o.reg===REG);
  $("k4").textContent = pr.length ? pr[0].dong : "없음";
  $("mapTitle").textContent = `${REG} ${LAYER==="heat"?"폭염":"호우"} 취약 상권 지도`;
  $("mapNote").textContent = act ? "격자 한 칸 = 50m. 마우스를 올리면 행정동·상권 유형·예상 감소율이 보입니다." :
    `이 날씨(${LAYER==="heat"?"폭염":"호우"})는 예보 기준에 들지 않아 모든 격자를 회색으로 표시합니다.`;
  // priority table
  if (!pr.length) $("prio").innerHTML = `<p>${REG==="강남" ? "예상 감소율 5% 이상인 격자가 없어 지원 대상이 없습니다 (모니터링)." : "예보 기준(강수 10mm·최고기온 33℃)에 들지 않아 지원 대상이 없습니다."}</p>`;
  else $("prio").innerHTML = "<table><tr><th>순위</th><th>행정동</th><th>날씨</th><th>고취약 칸</th><th>이중 취약</th><th>평균 예상 감소</th><th>지원</th></tr>" +
    pr.map((o,i)=>`<tr><td>${i+1}</td><td><b>${o.dong}</b><br><span class="muted">${o.top}${o.gl?` · 골프·레고 ${o.gl}`:""}</span></td><td>${o.lv}</td><td class="num">${o.hi}</td><td class="num">${o.dbl}</td><td class="num">${o.avg.toFixed(1)}%</td><td style="font-size:12px">${SHORT[o.lv]}</td></tr>`).join("") + "</table>";
  // 07 forecast by type
  let h = "<table><tr><th>지역</th><th>상권 유형</th><th>격자</th><th>"+mm+"mm 예보 시</th></tr>";
  for (const reg of ["춘천","강남"]) {
    const cc = D.cells[reg], acc = {};
    for (let i=0;i<cc.x.length;i++){ const k=cc.type[i]; (acc[k] ||= [0,0]); acc[k][0]+=interp(i,cc,mm); acc[k][1]++; }
    Object.entries(acc).sort((a,b)=>a[1][0]/a[1][1]-b[1][0]/b[1][1]).forEach(([k,[s,n]]) => {
      const v = s/n; h += `<tr><td>${reg}</td><td>${D.types[k]}</td><td class="num">${n.toLocaleString()}</td><td class="num ${Math.abs(v)<5?'grayv':''}"><b>${fmt(v)}%</b></td></tr>`;
    });
  }
  $("fc").innerHTML = h + "</table>";
  legend(); drawMap();
}
$("cRain").oninput = drawAll; $("cTmax").oninput = drawAll;
document.querySelectorAll("#regSeg button").forEach(b => b.onclick = () => { REG=b.dataset.v; if(REG==="강남") ZOOM="all"; drawAll(); });
document.querySelectorAll("#laySeg button").forEach(b => b.onclick = () => { LAYER=b.dataset.v; layerTouched=true; drawAll(); });
document.querySelectorAll("#zoomSeg button").forEach(b => b.onclick = () => { ZOOM=b.dataset.v; drawAll(); });
window.addEventListener("resize", () => { if (!$("viewC").classList.contains("hide")) drawMap(); });
</script>
</body>
</html>
"""
html = TEMPLATE.replace("__DATA__", json.dumps(DATA, ensure_ascii=False, separators=(",", ":")))
path = OUT / "dashboard.html"
path.write_text(html, encoding="utf-8")
print("저장:", path, f"({path.stat().st_size/1e6:.1f} MB)")


## 3. 미리 보기 (선택)
Colab 출력 칸에서 바로 확인한다. 실제 사용은 내려받은 파일을 브라우저로 여는 것을 권장.

In [ ]:
from IPython.display import HTML, display
display(HTML(open(OUT / "dashboard.html", encoding="utf-8").read()))

# 내 컴퓨터로 내려받기 (필요할 때만 주석 해제)
# from google.colab import files
# files.download(str(OUT / "dashboard.html"))
